In [1]:
import os
os.chdir('/workspace/70af3d91-ff91-484d-b06f-bba3d5ed2e13')
print(os.listdir('.'))


TimeoutError: Code execution timed out after 900.0 seconds

In [ ]:
# KERNEL_RESET

In [1]:
import os, subprocess
result = subprocess.run(['ls', '-la'], capture_output=True, text=True, cwd='/workspace/70af3d91-ff91-484d-b06f-bba3d5ed2e13')
print(result.stdout)
print("STDERR:", result.stderr)


total 731
drwxrwsr-x 5 root jupytergroup 4096 May 31 00:13 .
drwxr-xr-x 5 root root 4096 May 30 23:57 ..
drwxrwsr-x 5 root jupytergroup 4096 May 30 23:57 .config
-rw-rw-r-- 1 kerneluser jupytergroup 0 May 31 00:13 .kernel_llm_logs_1.txt
drwxrwsr-x 2 root jupytergroup 4096 May 30 23:57 .prompts
-rw-r--r-- 1 root jupytergroup 25848 May 30 23:57 -PROMPT-v6-DATASET.md
-rw-r--r-- 1 root jupytergroup 1577 May 30 23:57 delta_shard_runner.py
-rw-r--r-- 1 root jupytergroup 4649 May 30 23:57 generate_ldelta.py
-rw-r--r-- 1 root jupytergroup 694748 May 30 23:57 ldelta_zeros_N3405_approx_dps38.npy
drwxr-sr-x 2 root jupytergroup 4096 May 30 23:57 memory

STDERR: 


In [2]:
import os
os.chdir('/workspace/70af3d91-ff91-484d-b06f-bba3d5ed2e13')
print(os.getcwd())


/workspace/70af3d91-ff91-484d-b06f-bba3d5ed2e13


In [3]:
import numpy as np
approx = np.load('ldelta_zeros_N3405_approx_dps38.npy')
print(approx.shape, approx.dtype)
print(approx[0], '...', approx[-1])
print(float(approx[0]), float(approx[-1]))


(3405,) <U51
9.2223793999211025222437671927434781355 ... 2199.5340780429643642638069007845685866
9.222379399921103 2199.534078042964


In [4]:
import subprocess
# Check python version, available libs, cores
r = subprocess.run(['python', '-c', 'import cypari2; print("cypari2 ok")'], capture_output=True, text=True)
print("cypari2:", r.stdout, r.stderr)
r = subprocess.run(['nproc'], capture_output=True, text=True)
print("nproc:", r.stdout)
r = subprocess.run(['python', '-c', 'import mpmath; print(mpmath.__version__)'], capture_output=True, text=True)
print("mpmath:", r.stdout, r.stderr)


cypari2: Traceback (most recent call last):
 File "<string>", line 1, in <module>
 import cypari2; print("cypari2 ok")
 ^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'cypari2'

nproc: 8

mpmath: 1.3.0
 


In [5]:
import subprocess
r = subprocess.run(['pip', 'install', 'cypari2', 'cysignals'], capture_output=True, text=True)
print(r.stdout[-2000:])
print("STDERR:", r.stderr[-2000:])



STDERR: 


In [6]:
import subprocess
# Quick check cypari2 actually initializes and we can build L(Delta)
code = """
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
t0=time.time()
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
print("L built", time.time()-t0)
# test small lfunzeros
t0=time.time()
pari("Z = lfunzeros(L, [0, 30])")
n = int(pari("#Z"))
print("found",n,"zeros in <=30 in",time.time()-t0,"s")
print(str(pari('Strprintf("%.80g", Z[1])')))
"""
r = subprocess.run(['python', '-c', code], capture_output=True, text=True, timeout=300)
print(r.stdout)
print("STDERR:", r.stderr[-2000:])


PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
L built 0.0289762020111084
found 8 zeros in <=30 in 0.029797077178955078 s
9.2223793999211025222437671927434781355287706224320092899981864280019212953707290

STDERR: 


In [7]:
import subprocess
# Now test refinement via lfunhardy. The idea is: given an approximate gamma at dps=38, refine to dps=80 by finding root of lfunhardy near gamma.
# PARI's lfunhardy(L, t) returns the Z-function value; zeros of Z(t) on real line correspond to zeros of L(1/2+it).
# We can refine using solve() in PARI: solve(t=t_lo, t_hi, lfunhardy(L,t)) for a bracket around gamma.
code = """
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
# Approx gamma at dps~38 for first zero of L(Delta,s):
g = '9.2223793999211025222437671927434781355'
# Refine: bracket gamma +/- 1e-6, but need sign change
pari(f"g0 = {g}")
# Try bracket of width 0.001
for w in [1e-3, 1e-4, 1e-5]:
 a = pari(f"g0 - {w}")
 b = pari(f"g0 + {w}")
 fa = pari(f"lfunhardy(L, g0 - {w})")
 fb = pari(f"lfunhardy(L, g0 + {w})")
 print("w=",w,"fa=",str(fa)[:30],"fb=",str(fb)[:30])
t0=time.time()
res = pari("solve(t = g0-0.001, g0+0.001, lfunhardy(L, t))")
print("solve result:", str(pari(f'Strprintf("%.80g", {res})')))
print("elapsed",time.time()-t0)
"""
r = subprocess.run(['python', '-c', code], capture_output=True, text=True, timeout=300)
print(r.stdout)
print("STDERR:", r.stderr[-2000:])


PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
w= 0.001 fa= 2.0662730550178687894583915817 fb= -2.066789179589926519104251943
w= 0.0001 fa= 2.0665054040312399584378565668 fb= -2.066557016493393250820737023
w= 1e-05 fa= 2.0665286305687906503779402088 fb= -2.066533791815010927135750160
solve result: 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290
elapsed 0.008995771408081055

STDERR: 


In [8]:
import subprocess
# Excellent. Refinement works fast (~9ms) for first zero with bracket width 1e-3.
# Now test at a higher height where things are slower. Test gamma ~ 2199 (last in approx list).
code = """
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")

# Test at various heights
import numpy as np
arr = np.load('/workspace/70af3d91-ff91-484d-b06f-bba3d5ed2e13/ldelta_zeros_N3405_approx_dps38.npy')
for idx in [0, 100, 500, 1000, 2000, 3000, 3404]:
 g = str(arr[idx])
 pari(f"g0 = {g}")
 # Find a bracket: try shrinking widths
 t0=time.time()
 bracket_w = None
 for w in [1e-2, 1e-3, 1e-4, 1e-5]:
 fa = float(pari(f"lfunhardy(L, g0 - {w})"))
 fb = float(pari(f"lfunhardy(L, g0 + {w})"))
 if fa*fb < 0:
 bracket_w = w
 break
 if bracket_w is None:
 print(f"idx={idx} g={g[:30]} NO BRACKET")
 continue
 res = pari(f"solve(t = g0-{bracket_w}, g0+{bracket_w}, lfunhardy(L, t))")
 r = str(pari(f'Strprintf("%.80g", {res})'))
 elapsed = time.time()-t0
 print(f"idx={idx} g_approx={g[:25]:25s} w={bracket_w} elapsed={elapsed:.3f}s")
 print(f" refined ={r[:80]}")
"""
r = subprocess.run(['python', '-c', code], capture_output=True, text=True, timeout=600)
print(r.stdout)
print("STDERR:", r.stderr[-2000:])


PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
idx=0 g_approx=9.22237939992110252224376 w=0.01 elapsed=0.018s
 refined =9.222379399921102522243767192743478135528770622432009289998186428001921295370729
idx=100 g_approx=144.354726369403124051718 w=0.01 elapsed=0.075s
 refined =144.3547263694031240517189991331729492018921186483109652987204991162367937576925
idx=500 g_approx=471.276276566356679582964 w=0.01 elapsed=0.688s
 refined =471.2762765663566795829647331549277925057629596841655757358193812830220020873823
idx=1000 g_approx=811.807797486255547525606 w=0.01 elapsed=2.558s
 refined =811.8077974862555475256066984748699410809512731534842177294582335282641959459903
idx=2000 g_approx=1419.70566673416287196762 w=0.01 elapsed=12.637s
 refined =1419.705666734162871967626867623685848019439008008168826510187873151080463796824
idx=3000 g_approx=1981.28640846649667537917 w=0.01 elapsed=27.537s
 refined =1981.28640846649667537917860495391949705807998134880904360591761468313

In [9]:
# Estimate total cost for refinement of all 3405 zeros.
# Empirical: T^2.1 scaling. Use a few datapoints to estimate constant.
# At T=144, t=0.075s ; T=471, t=0.688 ; T=811, t=2.56 ; T=1419, t=12.6 ; T=1981, t=27.5 ; T=2199, t=36.3
import numpy as np
T = np.array([144.35, 471.28, 811.81, 1419.7, 1981.29, 2199.53])
t = np.array([0.075, 0.688, 2.558, 12.637, 27.537, 36.322])
# Fit log t = a + b log T
coef = np.polyfit(np.log(T), np.log(t), 1)
print("slope (exponent):", coef[0], " intercept:", coef[1])
# Predict total cost: integrate density of zeros * cost(T) over T.
# Zero density for L(Delta) ~ (deg/2pi) * log(T/2pi) ~ (2/2pi) * log(T/2pi) = log(T/2pi)/pi
# Total refinement cost = sum over zeros = approximately integrate density * c*T^b
# But we have actual list - just predict each zero from the fit.
c = np.exp(coef[1]); b = coef[0]
arr = np.load('ldelta_zeros_N3405_approx_dps38.npy')
gammas = np.array([float(x) for x in arr])
predicted = c * gammas**b
total_3405 = predicted.sum()
print(f"Predicted total refinement time for 3405 zeros (1 core): {total_3405:.0f}s = {total_3405/3600:.2f}h")
print(f"With 8 cores: {total_3405/8/3600:.2f}h")

# Need to extend to T~4100 for N=5000. Density: ~log(T/2pi)/pi
# Number of zeros up to T: N(T) ~ (T/2pi)log(T/2pi) - T/2pi for degree 2
from math import log, pi
def Nest(T): return T/(2*pi)*(log(T/(2*pi)) - 1)
print(f"N(2200) ≈ {Nest(2200):.0f}")
print(f"N(4100) ≈ {Nest(4100):.0f}")
# Approximate T for 5000th zero by inverting
import scipy.optimize as opt
T5000 = opt.brentq(lambda T: Nest(T)-5000, 100, 10000)
print(f"T(5000) ≈ {T5000:.1f}")
T3405 = opt.brentq(lambda T: Nest(T)-3405, 100, 10000)
print(f"T(3405) ≈ {T3405:.1f} (actual last={gammas[-1]:.1f})")


slope (exponent): 2.30965227635634 intercept: -14.30453957764273
Predicted total refinement time for 3405 zeros (1 core): 37821s = 10.51h
With 8 cores: 1.31h
N(2200) ≈ 1701
N(4100) ≈ 3576


T(5000) ≈ 5449.1
T(3405) ≈ 3933.2 (actual last=2199.5)


In [10]:
# Critical observation: density estimate N(T) ~ (T/2pi)log(T/2pi)-T/2pi gives N(2200)~1701
# but actual count is 3405. The dataset's density seems to be using "analytical normalization".
# Let's check actual density numerically.
print(f"Actual: 3405 zeros up to T={gammas[-1]:.2f}")
print(f"Density: {3405/gammas[-1]:.3f} zeros per unit T")
# Density formula: for L(Delta) (degree 2 modular form weight 12)
# N(T) ~ (T/pi) log(T) - (T/pi)(1+log(2pi)) + O(log T) for arithmetic normalization?
# Looking at LMFDB convention - for modular forms of weight k, central character, the 
# zero counting formula in analytic normalization is:
# N(T) = (T/pi)*log(T/(2*pi*e)) + O(log T) for degree 2
# Hmm actually for L(Delta,s) in analytic normalization (critical line Re(s)=1/2),
# N(T) ~ (T/pi)*log(T/(2*pi*e)) so with T=2200:
from math import log,pi,e
def Nest2(T): return T/pi * log(T/(2*pi*e))
print(f"N(2200)_v2 ≈ {Nest2(2200):.0f}")
print(f"N(4100)_v2 ≈ {Nest2(4100):.0f}")
# Try yet another formula
def Nest3(T): return T/pi * log(T/(2*pi))
print(f"N(2200)_v3 ≈ {Nest3(2200):.0f}")
print(f"N(4100)_v3 ≈ {Nest3(4100):.0f}")


Actual: 3405 zeros up to T=2199.53
Density: 1.548 zeros per unit T
N(2200)_v2 ≈ 3402
N(4100)_v2 ≈ 7153
N(2200)_v3 ≈ 4102
N(4100)_v3 ≈ 8458


In [11]:
# N_v2(2200) ≈ 3402 matches actual 3405 - great. So density ~(1/pi) log(T/(2 pi e))
# Now invert to find T(5000):
import scipy.optimize as opt
T5000 = opt.brentq(lambda T: Nest2(T)-5000, 100, 10000)
print(f"T(5000) ≈ {T5000:.1f}")
# Should be ~ 3000?
T_test = 3000
print(f"N(3000) ≈ {Nest2(3000):.0f}")
T_test = 3100
print(f"N(3100) ≈ {Nest2(3100):.0f}")


T(5000) ≈ 3032.8
N(3000) ≈ 4936
N(3100) ≈ 5132


In [12]:
# So we need search from T=2200 to roughly T=3050-3100 to find approximately 1600 more zeros.
# Total target: 5000 zeros. Refinement of all 5000 needed.

# Rough total cost: 
# Refine 3405 existing zeros (up to T=2200): ~37800s/8core = ~1.3h
# Search interval [2200, 3100] at dps=38 (much cheaper): partition with T^2.85 cost
# Refine ~1600 new zeros (T~2200 to 3100): each takes more than at T=2200, scaling T^2.3

# Estimate refinement of zeros from T=2200 to T=3100
# Already at T=2200, ~36s/zero. At T=3100, t = 36 * (3100/2200)^2.31 ≈ 36 * 2.1 = ~75s
# Average ~55s/zero * 1600 zeros = 88000s / 8 cores = ~3h
# So total: ~4-5h for refinement. Plus search ~1h.

# Compute budget: I have 3600s remaining (was 3600 at start, now ~2900s left after setup)
# This is INSUFFICIENT for full 5000 zero list.
# I need to be honest about this limitation per Rule 3 (limitations) and R8 (shrink N).

# Let me check remaining time and decide a strategy.
import time
print("Current time:", time.time())


Current time: 1780186747.216938


In [13]:
# Time budget: 3600s total runtime allowed. Already used substantial time.
# Strategy: Be pragmatic. Build the refinement script (production-quality), demonstrate it works,
# and execute the largest fraction of refinements feasible in remaining budget.
# At the end, transparently report: how many zeros refined, time used, and what remains.

# Plan:
# (a) Write refine_shard_runner.py that takes a list of (idx, gamma_approx_str) pairs,
# refines each via lfunhardy + solve, writes checkpoint per-zero (in case killed).
# (b) Write a driver refine_ldelta.py that partitions 3405 approx zeros into 8 shards
# by cost-aware (T^2.31) bins, launches 8 subprocess.Popen, and monitors.
# (c) Run for as long as we can afford (say ~2000s), then collect whatever was produced.
# (d) Also produce a search/extension script for new zeros in [2200, T_5000] but ackowledge
# we likely can't complete it.

# Memory note (R3): L(Delta,s) is the frozen control - LMFDB 1.12.a.a (modular form weight 12, level 1).
# Use exact construction lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1]).

# Let's first build the refinement worker.
refine_worker = r"""#!/usr/bin/env python3
'''refine_shard_runner.py - refine a list of approximate L(Delta,s) zeros to dps=80 via lfunhardy+solve.

Each refined zero is appended to a per-shard text file (one zero per line) as soon as it is
computed, so we get a usable checkpoint even if the process is killed mid-run.

Usage: python refine_shard_runner.py <idx> <approx_pkl> <dps> <out_txt>
 approx_pkl: pickle containing list of (orig_index, approx_gamma_string)
 out_txt: line-buffered append-only file written as: "<orig_index>\t<refined_str>\t<elapsed_s>\n"
'''
import sys, pickle, time, os
import cypari2

def main():
 if len(sys.argv) != 5:
 print('usage: refine_shard_runner.py idx approx_pkl dps out_txt', file=sys.stderr)
 sys.exit(2)
 idx = int(sys.argv[1])
 approx_pkl = sys.argv[2]
 dps = int(sys.argv[3])
 out_txt = sys.argv[4]

 with open(approx_pkl, 'rb') as f:
 items = pickle.load(f) # list[(orig_index, approx_str)]

 # Skip already-completed if checkpoint exists
 done = set()
 if os.path.exists(out_txt):
 with open(out_txt) as f:
 for line in f:
 if not line.strip():
 continue
 try:
 done.add(int(line.split('\t')[0]))
 except Exception:
 pass

 pari = cypari2.Pari()
 pari.default('realprecision', dps)
 pari.allocatemem(10**9)
 pari('L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])')

 out_f = open(out_txt, 'a', buffering=1)
 fmt = f'%.{dps}g'

 for orig_idx, approx in items:
 if orig_idx in done:
 continue
 t0 = time.time()
 try:
 pari(f'g0 = {approx}')
 # Try a sequence of bracket widths; lfunhardy is sign-change at each zero.
 bracket = None
 for w in (1e-2, 1e-3, 5e-4, 1e-4, 5e-5):
 fa = float(pari(f'lfunhardy(L, g0 - {w})'))
 fb = float(pari(f'lfunhardy(L, g0 + {w})'))
 if fa * fb < 0:
 bracket = w
 break
 if bracket is None:
 out_f.write(f'{orig_idx}\tERROR_NO_BRACKET\t{time.time()-t0:.3f}\n')
 continue
 pari(f'r = solve(t = g0 - {bracket}, g0 + {bracket}, lfunhardy(L, t))')
 refined = str(pari(f'Strprintf(\"{fmt}\", r)'))
 except Exception as e:
 out_f.write(f'{orig_idx}\tERROR_{type(e).__name__}\t{time.time()-t0:.3f}\n')
 continue
 elapsed = time.time() - t0
 out_f.write(f'{orig_idx}\t{refined}\t{elapsed:.3f}\n')

 out_f.close()
 print(f'[shard {idx}] done', flush=True)

if __name__ == '__main__':
 main()
"""

with open('refine_shard_runner.py', 'w') as f:
 f.write(refine_worker)
print("Wrote refine_shard_runner.py", len(refine_worker), "bytes")


Wrote refine_shard_runner.py 2721 bytes


In [14]:
import numpy as np, pickle, os
# Load approximate zeros
arr = np.load('ldelta_zeros_N3405_approx_dps38.npy')
print("Loaded:", arr.shape, arr.dtype)

# Cost-aware partition: per-zero cost ~ T^2.31 from our fit.
# We want each shard to have roughly equal total cost.
# Sort zeros and assign greedily.
gammas = np.array([float(s) for s in arr])
b = 2.31
costs = gammas**b
print(f"Total cost units: {costs.sum():.3e}")

NUM_SHARDS = 8
target_per = costs.sum() / NUM_SHARDS

# Greedy bucketing: assign zeros in ORDER (preserve original order) but split into contiguous chunks of roughly equal cost.
# Actually for better load balance, use ROUND-ROBIN by cost: sort by descending cost, assign to lightest shard.
# But we need stable mapping so checkpoint files are stable.
# Let's do: assign zero i to shard i % NUM_SHARDS, since cumulative cost is dominated by high-T zeros.
# Even better: stride-by-shard with reverse sort.

# Sort by cost descending, then assign to lightest shard
order = np.argsort(-costs) # descending
shards = [[] for _ in range(NUM_SHARDS)]
shard_cost = [0.0]*NUM_SHARDS
for idx in order:
 s = int(np.argmin(shard_cost))
 shards[s].append(int(idx))
 shard_cost[s] += costs[idx]

print("Shard sizes:", [len(s) for s in shards])
print("Shard cost units:", [f"{c:.3e}" for c in shard_cost])

# Save approx pickle files per shard
os.makedirs('ldelta_refine_shards', exist_ok=True)
for sidx, idx_list in enumerate(shards):
 # sort within shard by orig_idx for sanity
 idx_list_sorted = sorted(idx_list)
 items = [(i, str(arr[i])) for i in idx_list_sorted]
 with open(f'ldelta_refine_shards/approx_shard_{sidx}.pkl','wb') as f:
 pickle.dump(items, f)
print("Wrote shard pickle files")


Loaded: (3405,) <U51
Total cost units: 6.184e+10
Shard sizes: [426, 426, 426, 425, 426, 425, 425, 426]
Shard cost units: ['7.729e+09', '7.729e+09', '7.729e+09', '7.729e+09', '7.729e+09', '7.729e+09', '7.729e+09', '7.729e+09']
Wrote shard pickle files


In [15]:
# Predict per-shard wall time given the c constant from fit
import numpy as np
c = np.exp(-14.3045)
# Each shard ~ 7.73e9 cost units * c
predicted_per_shard = 7.729e9 * c
print(f"Predicted per-shard wall time: {predicted_per_shard:.0f}s = {predicted_per_shard/60:.1f} min")
# = total cost / 8 cores


Predicted per-shard wall time: 4740s = 79.0 min


In [16]:
# 79 min per shard with 8 parallel - too long for 3600s remaining budget.
# Reality check: my fit was based on a few points. Let's measure 100 zeros sampled across height range serially first
# to get a better cost estimate.

# But pragmatically: launch all 8 shards now and let them run. Periodically poll. After ~30 min, kill them
# and assemble whatever was completed. That preserves checkpoints (text files written line-by-line).

import subprocess, sys, os, time

NUM_SHARDS = 8
procs = []
for sidx in range(NUM_SHARDS):
 approx_pkl = f'ldelta_refine_shards/approx_shard_{sidx}.pkl'
 out_txt = f'ldelta_refine_shards/refined_shard_{sidx}.tsv'
 log = open(f'ldelta_refine_shards/shard_{sidx}.log', 'w')
 cmd = [sys.executable, 'refine_shard_runner.py', str(sidx), approx_pkl, '80', out_txt]
 p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
 procs.append((p, log, out_txt, sidx))
print(f"Launched {NUM_SHARDS} subprocesses, PIDs:", [p.pid for p,_,_,_ in procs])
launch_time = time.time()


Launched 8 subprocesses, PIDs: [203, 204, 205, 206, 207, 208, 209, 210]


In [17]:
# Poll every minute, watch progress. After ~20 min, decide on stopping.
import time, os, glob
def status():
 total_done = 0
 per_shard = []
 for sidx in range(8):
 f = f'ldelta_refine_shards/refined_shard_{sidx}.tsv'
 n = 0
 if os.path.exists(f):
 with open(f) as fh:
 for _ in fh: n += 1
 per_shard.append(n)
 total_done += n
 return total_done, per_shard

t0 = launch_time
# Print initial status
time.sleep(15)
t, ps = status()
elapsed = time.time()-t0
print(f"t={elapsed:.0f}s done={t} per_shard={ps}")


t=22s done=447 per_shard=[56, 56, 57, 55, 56, 55, 56, 56]


In [18]:
import time
# Wait longer.
time.sleep(60)
t, ps = status()
elapsed = time.time()-launch_time
print(f"t={elapsed:.0f}s done={t}/3405 ({100*t/3405:.1f}%) per_shard={ps}")


t=88s done=761/3405 (22.3%) per_shard=[95, 96, 96, 95, 95, 94, 95, 95]


In [19]:
import time
time.sleep(120)
t, ps = status()
elapsed = time.time()-launch_time
print(f"t={elapsed:.0f}s done={t}/3405 ({100*t/3405:.1f}%) per_shard={ps}")
# rate
print(f"rate: {t/elapsed:.2f} zeros/sec, ETA for 3405: {(3405-t)/(t/elapsed):.0f}s more")


t=215s done=1038/3405 (30.5%) per_shard=[130, 130, 130, 129, 130, 129, 129, 131]
rate: 4.84 zeros/sec, ETA for 3405: 489s more


In [20]:
import time
# Rate is decreasing as we hit higher T zeros. Continue polling.
time.sleep(180)
t, ps = status()
elapsed = time.time()-launch_time
print(f"t={elapsed:.0f}s done={t}/3405 ({100*t/3405:.1f}%) per_shard={ps}")
print(f"rate: {t/elapsed:.2f} zeros/sec")


t=401s done=1282/3405 (37.7%) per_shard=[161, 160, 161, 160, 160, 159, 160, 161]
rate: 3.19 zeros/sec


In [21]:
import time
time.sleep(180)
t, ps = status()
elapsed = time.time()-launch_time
print(f"t={elapsed:.0f}s done={t}/3405 ({100*t/3405:.1f}%) per_shard={ps}")
print(f"rate: {t/elapsed:.2f} zeros/sec")
# while we wait, start preparing search extension scripts


t=588s done=1448/3405 (42.5%) per_shard=[182, 182, 181, 180, 181, 180, 181, 181]
rate: 2.46 zeros/sec


In [22]:
# While refinement runs, build the search extension worker for T in [2200, 3100].
# Use the same delta_shard_runner.py pattern but at dps=38 only.

import time
# Estimate when refinement will finish.
# Each shard is processing in orig_idx order; cost-aware bins all have approx equal total work.
# Rate decreasing means we're hitting higher T. Per-shard work: 426 zeros with total cost 7.7e9 units.
# Avg cost ~1.8e7 per zero -> avg time per zero (single-shard) = c*1.8e7 = exp(-14.3)*1.8e7 = 11s.
# 426 * 11s = 4690s. Cumulative within shard: lots of fast small ones early, slow ones late.

# Look at TSV files - what's the timing pattern?
import os
sizes = []
for sidx in range(8):
 f = f'ldelta_refine_shards/refined_shard_{sidx}.tsv'
 with open(f) as fh:
 lines = fh.readlines()
 if lines:
 last = lines[-1].strip().split('\t')
 first = lines[0].strip().split('\t')
 last_idx = int(last[0])
 last_t = float(last[2])
 sizes.append((sidx, len(lines), last_idx, last_t))
for s in sizes:
 print(s)


(0, 183, 1459, 10.091)
(1, 182, 1448, 9.988)
(2, 183, 1454, 10.475)
(3, 181, 1450, 10.074)
(4, 183, 1460, 9.942)
(5, 181, 1452, 9.949)
(6, 182, 1458, 10.267)
(7, 183, 1456, 9.903)


In [23]:
import time
# Each shard at ~183/426 zeros done, currently each zero takes ~10s. Last zeros will take ~50s.
# Remaining ~243 zeros per shard, gradually slower. Estimate via cost-aware reasoning:
# Each shard has equal total cost 7.7e9. Done so far ~ partial. Let's actually check shard cost progress.

# Skip detailed estimate; just continue and stop in another ~15 min total.
time.sleep(180)
t, ps = status()
elapsed = time.time()-launch_time
print(f"t={elapsed:.0f}s done={t}/3405 ({100*t/3405:.1f}%) per_shard={ps}")
print(f"rate: {t/elapsed:.2f} zeros/sec")


t=788s done=1590/3405 (46.7%) per_shard=[200, 199, 199, 198, 199, 198, 198, 199]
rate: 2.02 zeros/sec


In [24]:
import time
# Continue polling. Let's wait until ~1400s elapsed then check.
time.sleep(300)
t, ps = status()
elapsed = time.time()-launch_time
print(f"t={elapsed:.0f}s done={t}/3405 ({100*t/3405:.1f}%) per_shard={ps}")
print(f"rate: {t/elapsed:.2f} zeros/sec")
# Look at TSV timing pattern - last 10 lines of shard 0
with open('ldelta_refine_shards/refined_shard_0.tsv') as fh:
 lines = fh.readlines()
print("Last 5 of shard 0:")
for ln in lines[-5:]:
 p = ln.strip().split('\t')
 print(f" idx={p[0]} t={p[2]}s")


t=1096s done=1765/3405 (51.8%) per_shard=[221, 221, 221, 219, 221, 220, 220, 222]
rate: 1.61 zeros/sec
Last 5 of shard 0:
 idx=1732 t=14.763s
 idx=1735 t=14.696s
 idx=1746 t=15.147s
 idx=1752 t=15.131s
 idx=1761 t=15.652s


In [25]:
import time
# 1178s remaining. Need to plan termination. Let refinement run ~600 more seconds, then kill and assemble.
# Continue polling more carefully.
time.sleep(300)
t, ps = status()
elapsed = time.time()-launch_time
print(f"t={elapsed:.0f}s done={t}/3405 ({100*t/3405:.1f}%) per_shard={ps}")


t=1408s done=1907/3405 (56.0%) per_shard=[239, 239, 239, 237, 238, 238, 238, 239]


In [26]:
import time, os, signal
# 866s left. Need to: poll once more briefly, then terminate, assemble what we have, save final partial product.
# Wait another 400s (8 cores still working, each zero ~25s now -> +128 zeros expected).
time.sleep(400)
t, ps = status()
elapsed = time.time()-launch_time
print(f"t={elapsed:.0f}s done={t}/3405 ({100*t/3405:.1f}%) per_shard={ps}")


TimeoutError: Code execution timed out after 258.0 seconds